# Can we predict `arrears_next_month`?
Follows `checklist.md`. Key decisions:
- **Leakage:** `final_notice_sent` is recorded in the cycle *after* the billing month, so it is not known at prediction time (and is almost a proxy for the target). Dropped. `meter_ref` is an identifier: used only to group rows, never as a feature. `billing_month` is used only for ordering/splitting.
- **Split:** the model is meant for future rows, and accounts repeat. So the test set = a random 20% of *whole accounts*, restricted to the *latest 4 months*. Training = other accounts, only months before the test window.
- **CV:** inside training only, forward in time by month, with validation accounts held out of that fold's training rows.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss
from sklearn.base import clone
df = pd.read_csv('data.csv', parse_dates=['billing_month'])
df.shape, df.arrears_next_month.mean()

## 1. Test set, set aside before any modelling

In [ ]:
rng = np.random.RandomState(0)
accounts = df.meter_ref.unique()
test_accts = set(rng.choice(accounts, size=int(0.2*len(accounts)), replace=False))
months = np.sort(df.billing_month.unique())
cutoff = months[-4]                      # test window = last 4 months
is_test = df.meter_ref.isin(test_accts) & (df.billing_month >= cutoff)
is_train = ~df.meter_ref.isin(test_accts) & (df.billing_month < cutoff)
train, test = df[is_train].reset_index(drop=True), df[is_test].reset_index(drop=True)
assert not set(train.meter_ref) & set(test.meter_ref) and train.billing_month.max() < test.billing_month.min()
print(len(train), len(test), train.arrears_next_month.mean(), test.arrears_next_month.mean())
# `test` is not touched again until section 5.

## 2. Features (known at prediction time only)
`tariff, occupants, kwh, bill_eur, late_payments_12m` all describe the billing month. `final_notice_sent` is excluded (post-billing); identifiers are excluded.

In [ ]:
CAT, NUM = ['tariff'], ['occupants','kwh','bill_eur','late_payments_12m']
FEATS = CAT + NUM; TARGET = 'arrears_next_month'
def prep(): return ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), CAT), ('n', StandardScaler(), NUM)])

## 3. Candidates (named before any scoring)
1. `baseline`: predicts the training prevalence (trivial).
2. `logreg`: logistic regression, C in {0.1, 1, 10}.
3. `hgb`: gradient boosting, max_depth in {2, 3}, learning_rate 0.05.

Selection metric: CV average precision (PR-AUC). The decision threshold is chosen on CV out-of-fold predictions to maximise F1.
Scaling/encoding live inside the pipeline, so they are fitted per fold.

In [ ]:
candidates = {'baseline': Pipeline([('p',prep()),('m',DummyClassifier(strategy='prior'))])}
for C_ in [0.1,1,10]:
    candidates[f'logreg_C{C_}'] = Pipeline([('p',prep()),('m',LogisticRegression(C=C_,max_iter=1000))])
for d in [2,3]:
    candidates[f'hgb_d{d}'] = Pipeline([('p',prep()),('m',HistGradientBoostingClassifier(max_depth=d,learning_rate=0.05,random_state=0))])
list(candidates)

## 4. Cross-validation inside training rows (forward in time, whole accounts held out)

In [ ]:
tm = np.sort(train.billing_month.unique())
def folds(n_folds=4, val_len=3):
    # validation = block of consecutive months; train = earlier months, accounts not in that fold's validation set
    out = []
    accts = train.meter_ref.unique()
    for k in range(n_folds):
        end = len(tm) - (n_folds-1-k)*val_len
        v0 = tm[end-val_len]; v1 = tm[end-1]
        r = np.random.RandomState(100+k)
        vacc = set(r.choice(accts, size=len(accts)//4, replace=False))
        va = train.meter_ref.isin(vacc) & train.billing_month.between(v0, v1)
        tr = ~train.meter_ref.isin(vacc) & (train.billing_month < v0)
        out.append((np.where(tr)[0], np.where(va)[0]))
    return out
FOLDS = folds()
for tr,va in FOLDS: print(len(tr), len(va), train.billing_month[tr].max().date(), train.billing_month[va].min().date())
oof = {}
for name, mdl in candidates.items():
    preds = []
    for tr,va in FOLDS:
        m = clone(mdl).fit(train.loc[tr,FEATS], train.loc[tr,TARGET])
        preds.append(pd.Series(m.predict_proba(train.loc[va,FEATS])[:,1], index=va))
    oof[name] = pd.concat(preds)
y_oof = train.loc[oof['baseline'].index, TARGET]
res = pd.DataFrame({n:{'cv_pr_auc':average_precision_score(y_oof,p),'cv_roc_auc':roc_auc_score(y_oof,p)} for n,p in oof.items()}).T
res.sort_values('cv_pr_auc', ascending=False)

In [ ]:
best = res.cv_pr_auc.idxmax()
ths = np.linspace(0.05,0.95,91)
f1s = [f1_score(y_oof, oof[best]>=t) for t in ths]
THR = float(ths[int(np.argmax(f1s))])
print('chosen model:', best, '| threshold:', round(THR,2), '| CV F1:', round(max(f1s),3))

## 5. Final choice (written down before touching the test set)
Chosen from CV only: the model and threshold printed above (highest CV PR-AUC). It is refit on all training rows and scored **once** on the test set.

In [ ]:
final = clone(candidates[best]).fit(train[FEATS], train[TARGET])
p = final.predict_proba(test[FEATS])[:,1]
base = clone(candidates['baseline']).fit(train[FEATS], train[TARGET]).predict_proba(test[FEATS])[:,1]
y = test[TARGET]
pd.Series({'model':best,'threshold':THR,'test_pr_auc':average_precision_score(y,p),'test_roc_auc':roc_auc_score(y,p),
           'test_f1':f1_score(y,p>=THR),'test_brier':brier_score_loss(y,p),
           'prevalence (=baseline PR-AUC)':y.mean(),'baseline_roc_auc':roc_auc_score(y,base)})

## 6. Conclusion
The headline is the test-set row above: a single, one-shot, out-of-time, unseen-account score. Compare PR-AUC with prevalence (what a no-skill model gets). Rows within an account are correlated, so treat the numbers as approximate. Because `final_notice_sent` was excluded, performance reflects what can be predicted at billing time.